#  Predicción sobre Test

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

### Cargamos los datos 

In [3]:
test = pd.read_csv(r"../data/test_clean.csv")
train = pd.read_csv(r"../data/train_clean_sin_outl.csv")

X = train.drop("Item_Outlet_Sales", axis=1)
y = train["Item_Outlet_Sales"]
X_test = test.reindex(columns=X.columns, fill_value=0)

### XGBOOST

In [4]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,      
    random_state=42     
)

xgb = XGBRegressor(       
    n_estimators=100,     
    learning_rate=0.05,    
    max_depth=4,           
    subsample=0.8,        
    colsample_bytree=0.8, 
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_val)
y_pred_xgb_train = xgb.predict(X_train)

rmse_xgb = np.sqrt(mean_squared_error(y_val, y_pred_xgb))
mae_xgb = mean_absolute_error(y_val, y_pred_xgb)
r2_xgb = r2_score(y_val, y_pred_xgb)
r2_xgb_train = r2_score(y_train, y_pred_xgb_train)
rmse_xgb_train = np.sqrt(mean_squared_error(y_train, y_pred_xgb_train))

print("XGBOOST")
print(f"RMSE Train: {rmse_xgb_train:.2f}  |  RMSE Val: {rmse_xgb:.2f}")
print(f"R2 Train:   {r2_xgb_train:.4f}   |  R2 Val:   {r2_xgb:.4f}")
print(f"                     |  MAE Val:  {mae_xgb:.2f}")

XGBOOST
RMSE Train: 905.92  |  RMSE Val: 912.75
R2 Train:   0.6123   |  R2 Val:   0.5989
                     |  MAE Val:  666.80


### Predicción

In [5]:
# Predicciones
Predicciones = xgb.predict(X_test)

resultados = pd.DataFrame({
    "Item_Outlet_Sales_Predicted": Predicciones
})

print(f"Predicciones generadas: {len(resultados)}")
print(resultados.head(10))

Predicciones generadas: 5681
   Item_Outlet_Sales_Predicted
0                  1687.203857
1                  1456.251709
2                   815.240845
3                  2560.226074
4                  4331.252441
5                  1845.339722
6                   613.771606
7                  2200.752930
8                  1523.972656
9                  3146.410156


### Resultados

In [7]:
test_final = pd.read_csv(r"../data/Test_BigMart.csv")
test_final["Item_Outlet_Sales_Prediccion"] = Predicciones
print(f"Predicciones: {Predicciones.shape}")
print(f"Original: {test_final.shape}")

Predicciones: (5681,)
Original: (5681, 12)


In [8]:
test_final.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales_Prediccion
0,FDW58,20.750,Low Fat,0.007565,Snack Foods,107.8622,OUT049,1999,Medium,Tier 1,Supermarket Type1,1687.203857
1,FDW14,8.300,reg,0.038428,Dairy,87.3198,OUT017,2007,NaN,Tier 2,Supermarket Type1,1456.251709
2,NCN55,14.600,Low Fat,0.099575,Others,241.7538,OUT010,1998,NaN,Tier 3,Grocery Store,815.240845
3,FDQ58,7.315,Low Fat,0.015388,Snack Foods,155.0340,OUT017,2007,NaN,Tier 2,Supermarket Type1,2560.226074
4,FDY38,NaN,Regular,0.118599,Dairy,234.2300,OUT027,1985,Medium,Tier 3,Supermarket Type3,4331.252441


In [ ]:
test_final.to_csv(r"../entregables/test_predicciones.csv", index=False)